In [1]:
from IPython.display import Image

- https://bigeagle.me/2025/07/kimi-k2/
    - 把人与AI的交互方式，从 chat-first 变成 artifact-first：你和 AI 交互的过程不是为了它直接输出一段内容，而是它理解用户的需求后 立刻开启一个小工程，交付一个前端应用出来，用户可以继续追问、修改、迭代，但这些都围绕着一份交付物进行。
        - https://aider.chat/
        - https://github.com/Aider-AI/aider
- tool calling and agentic loops, can call multiple tools in parallel and reliably, and knows when to stop

### Tool Use & Agent

- 当时我们在 K1.5 研发过程中通过 RLVR (Reinforcement Learning with Verifiable Rewards) 取得了相当不错的效果，就想着复刻这套方法，搞它一堆真实的 `MCP Server` 直接接进 RL 环境中联合训练。
    - 部署麻烦，例如 Blender MCP 对于已经有 blender 的用户很容易，但在 RL 环境中装上 blender 就是一个负担；其次也是更致命的，不少第三方工具需要登录使用，你总不能为了训练 Notion MCP 使用而去注册一堆 Notion 账号吧？
    - **模型在预训练中已经知道工具该怎么用了**，我们只需要把这个能力激发出来。这个假设的的基础很容易理解：
        - **预训练见过大量的代码数据**，其中有大量的、用各种语言和表达方式的 API call， 如果把每个 API call 都当成一种工具，那么模型早就该会用了
        - 另一个基础是，**预训练模型本身就掌握了丰富的世界知识**，比如你让他角色扮演一个 Linux Terminal，它完全能和你像模像样的交互一番， 那么显然对于 terminal tool 调用应当只需要少量数据就可以激发出来。
- 我们设计了一个比较精巧的 workflow，让模型自己合成海量的 Tool Spec 和使用场景，通过 multiagent 的方式合成了非常 diverse 的工具调用类数据，果然效果不错。

```python
task = get_user_input()
history = [task, ]
while True:
    resp = model(history, toolset)
    history.append(resp)
    if not resp.tool_calls:
        break

    for tool_call in tool_calls:
        result = call_tool(tool_call)
        history.append(result)
```

### GAIA => Alita

- mcp creation & self evolving
    - minimal predefinition
    - maximal self-evolution

In [3]:
Image(url='./figs/Alita.png', width=500)